# Módulo 3 — Análise de Dados · Notebook ETL

Bem-vindo(a) ao **notebook central** do Módulo 3. Aqui a regra da casa continua a mesma:

```
Pense o problema  →  dirija a IA  →  confira o resultado.
```

E o esqueleto que se repete agora é outro, mas igualmente simples:

```
ler a base  →  explorar a sujeira  →  limpar campo por campo  →  conferir  →  salvar a base limpa
```

Trabalhamos com **três bases do Piauí** (municípios, escolas, servidores), unidas depois por **Código IBGE**:
- `municipios_piaui_suja.xlsx`  — simples, com **latitude/longitude**
- `escolas_piaui_suja.xlsx`     — média (matrículas, dependência, situação)
- `servidores_piaui_suja.xlsx`  — complexa (datas, CPF, remuneração, duplicatas)

No fim gravamos **três planilhas limpas** em `saida/modulo3/`, que os scripts `m3_cruzamento.py`,
`m3_agrupamentos.py`, `m3_relatorio_reportlab.py` e o dashboard Streamlit vão consumir.

---

## Pensamento computacional aqui (os 4 pilares)
  • **Decomposição** → cada base é um bloco; cada tipo de sujeira é uma função pequena.
  • **Padrões**      → a sujeira se repete (IBGE com zeros, datas com vários separadores, caixa).
  • **Abstração**    → "limpar uma coluna numérica" vale p/ População, Matrículas e Remuneração.
  • **Algoritmo**    → ler → explorar → limpar campo por campo → conferir → salvar.


## 0 · Preparação (imports e caminhos)

Importamos pandas, matplotlib (com `Agg` para exportar PNG sem abrir janela) e definimos os
caminhos relativos à raiz do projeto. As bases **nunca** são alteradas — sempre lemos
de `entrada/modulo3/` e gravamos em `saida/modulo3/`.

In [ ]:
import re
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # exportar PNG sem abrir janela
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.figsize'] = (8, 5)

RAIZ = Path.cwd()
# ajuste se rodar de outra pasta: RAIZ deve ser a raiz do repositório
if not (RAIZ / 'entrada').exists():
    RAIZ = Path.cwd().parent.parent  # fallback quando rodando de scripts/modulo3/
ENTRADA = RAIZ / 'entrada' / 'modulo3'
SAIDA   = RAIZ / 'saida'   / 'modulo3'
GRAFICOS = SAIDA / 'graficos'
SAIDA.mkdir(parents=True, exist_ok=True)
GRAFICOS.mkdir(parents=True, exist_ok=True)
print('Entrada:', ENTRADA)
print('Saída  :', SAIDA)


## 1 · Carregar e EXPLORAR a sujeira

A mesma ideia da Frente 0 do Módulo 2: **olhar antes de mexer**. Em cada base, `head()` já
deixa aparecer os problemas (IBGE com zero à esquerda, latitude com vírgula, datas bagunçadas…).
É essa sujeira que vai criar a **necessidade** de limpar — e justificar cada função.

In [ ]:
df_mun = pd.read_excel(ENTRADA / 'municipios_piaui_suja.xlsx', dtype=str)
df_esc = pd.read_excel(ENTRADA / 'escolas_piaui_suja.xlsx', dtype=str)
df_ser = pd.read_excel(ENTRADA / 'servidores_piaui_suja.xlsx', dtype=str)

print('MUNICÍPIOS', df_mun.shape)
display(df_mun.head(4))
print('\nESCOLAS', df_esc.shape)
display(df_esc.head(4))
print('\nSERVIDORES', df_ser.shape)
display(df_ser.head(4))


> **Repare** (antes de limpar):
> - `Código_IBGE` aparece como `221100`, `0220770`, `221.100`, `0220490`… (zeros e pontuação).
> - `Latitude`/`Longitude` ora usam `.` ora `,` como separador decimal.
> - `População` às vezes vem como `'868000'`, às vezes como `'868000 hab'`.
> - `Matrículas` e `Remuneração` chegam como texto, com `'R$ '` e vírgula.
> - `Admissão` traz 4 formatos distintos (`2015-03-12`, `05/07/2018`, `12-11-2010`, `2020/01/30`).
> - Há duplicatas de CPF e linhas com município repetido.

---


## 2 · Limpezas — uma função por problema (decomposição)

Cada sujeira vira uma função pequena e isolada, exatamente como na Frente 0 do Módulo 2.
Reaproveite a ideia: **padronizar antes de comparar** (abstração).

### 2.1 Chave IBGE — o coração do cruzamento
Tudo se junta por `Código_IBGE`. Se ele não estiver num formato só, o `merge` quebra. Por isso
a primeira limpeza é **normalizar o IBGE** para um inteiro, descartando zeros à esquerda e pontos.

In [ ]:
def limpar_ibge(df, col='Código_IBGE'):
    """'0220770', '221.100', '000221100' -> 221100 (int) ou NaN se vazio."""
    s = (df[col].astype(str)
               .str.replace('.', '', regex=False)
               .str.replace('-', '', regex=False)
               .str.strip()
               .str.lstrip('0'))
    df[col] = pd.to_numeric(s, errors='coerce').astype('Int64')
    return df

df_mun = limpar_ibge(df_mun)
df_esc = limpar_ibge(df_esc)
df_ser = limpar_ibge(df_ser)
print('IBGE único em municípios:', df_mun['Código_IBGE'].nunique(), 'de', len(df_mun))
print(df_mun['Código_IBGE'].head(6).to_string())


### 2.2 Texto — espaços e caixa
Padroniza nomes de município, escola, cargo, situação e dependência.

In [ ]:
def limpar_texto(df, colunas):
    for col in colunas:
        df[col] = (df[col].astype('string')
                          .str.strip()
                          .str.split()
                          .str.join(' '))
    return df

df_mun = limpar_texto(df_mun, ['Município', 'Região'])
df_mun['Município'] = df_mun['Município'].str.title()
df_mun['Região']    = df_mun['Região'].str.title()

df_esc = limpar_texto(df_esc, ['Município', 'Nome_Escola', 'Dependência', 'Situação'])
df_esc['Dependência'] = df_esc['Dependência'].str.title()
df_esc['Município']    = df_esc['Município'].str.title()

df_ser = limpar_texto(df_ser, ['Nome', 'Cargo', 'Situação', 'Lotação'])
df_ser['Nome']  = df_ser['Nome'].str.title()
df_ser['Cargo'] = df_ser['Cargo'].str.title()
display(df_mun[['Município','Região']].head(3))


### 2.3 Números disfarçados de texto
População (`'868000 hab'`), Matrículas (`'488'`), Remuneração (`'R$ 4163,24'`).
O padrão é o mesmo: **tirar tudo que não é dígito/vírgula, trocar vírgula por ponto, virar float**.

In [ ]:
def limpar_numero_coluna(serie):
    """'R$ 4.163,24' / '868000 hab' -> float."""
    s = (serie.astype(str)
               .str.replace('R$', '', regex=False)
               .str.replace('hab', '', regex=False)
               .str.replace('.', '', regex=False)   # milhar
               .str.replace(',', '.', regex=False)  # decimal
               .str.strip())
    return pd.to_numeric(s, errors='coerce')

df_mun['População']   = limpar_numero_coluna(df_mun['População']).astype('Int64')
df_esc['Matrículas']  = limpar_numero_coluna(df_esc['Matrículas']).astype('Int64')
df_ser['Remuneração'] = limpar_numero_coluna(df_ser['Remuneração'])
display(df_mun[['Município','População']].head(3))
display(df_ser[['Nome','Remuneração']].head(3))


### 2.4 >>> EXERCÍCIO DOS ALUNOS — Latitude e Longitude <<<

As colunas `Latitude` e `Longitude` vêm com `,` como separador decimal numa base e `.` noutra.
**Sua tarefa:** deixar ambas como `float` numericamente consistentes.

> **Dica de prompt para a IA** (como ensina a aula):
> "Em pandas, converta as colunas Latitude e Longitude, que vêm como texto com separador
> decimal misturado (algumas com '.', outras com ','), para float. Valores que não derem para
> interpretar devem virar NaN, não quebrar o script."

Apague o `return df` abaixo e implemente. Depois, **confira** com `df_mun[['Latitude','Longitude']].dtypes`.

---


In [ ]:
def limpar_coords(df, cols=('Latitude', 'Longitude')):
    """TODO(aluno): normalizar Latitude/Longitude para float."""
    # Dica: troque ',' por '.' e use pd.to_numeric(..., errors='coerce').
    return df

df_mun = limpar_coords(df_mun)
df_mun[['Latitude', 'Longitude']].dtypes


### 2.5 Datas — o mesmo desafio da Frente 0 (Módulo 2)

`Admissão` chega em 4 formatos. Reaproveite a estratégia:
`pd.to_datetime(..., dayfirst=True, errors='coerce')` e depois `.dt.strftime('%d/%m/%Y')`.
Marcadores `NaT` viram `'(data inválida)'` para não sumir na planilha.

In [ ]:
def limpar_datas(df, col='Admissão'):
    dt = pd.to_datetime(df[col], dayfirst=True, errors='coerce')
    df[col] = dt.dt.strftime('%d/%m/%Y').fillna('(data inválida)')
    return df

df_ser = limpar_datas(df_ser)
display(df_ser[['Nome','Admissão']].head(5))


### 2.6 CPF e duplicatas — padronizar antes de comparar

Mesmo algoritmo da Frente 0: formatar em `000.000.000-00` e **só então** remover duplicatas.

In [ ]:
def limpar_cpf(df, col='CPF'):
    def fmt(cpf):
        d = ''.join(c for c in str(cpf) if c.isdigit())
        return f'{d[:3]}.{d[3:6]}.{d[6:9]}-{d[9:]}' if len(d) == 11 else f'INVÁLIDO({cpf})'
    df[col] = df[col].map(fmt)
    return df

antes = len(df_ser)
df_ser = limpar_cpf(df_ser)
df_ser = df_ser.drop_duplicates(subset='CPF', keep='first').reset_index(drop=True)
print(f'Servidores: {antes} -> {len(df_ser)} (removidas {antes - len(df_ser)} duplicatas)')


### 2.7 Remover municípios duplicados por IBGE
A base de municípios veio com uma linha repetida (mesmo Código IBGE). Mantemos a primeira.

In [ ]:
antes = len(df_mun)
df_mun = df_mun.drop_duplicates(subset='Código_IBGE', keep='first').reset_index(drop=True)
print(f'Municípios: {antes} -> {len(df_mun)}')
display(df_mun.head(12))


## 3 · Salvar as três bases limpas

Conferimos o resultado (parte do trabalho, slide 8) e gravamos em `saida/modulo3/`.
Esses arquivos são o combustível dos scripts `m3_*`.

In [ ]:
df_mun.to_excel(SAIDA / 'municipios_limpo.xlsx',   index=False)
df_esc.to_excel(SAIDA / 'escolas_limpo.xlsx',       index=False)
df_ser.to_excel(SAIDA / 'servidores_limpo.xlsx',    index=False)
print('Gravado:')
print(' • municipios_limpo.xlsx ', df_mun.shape)
print(' • escolas_limpo.xlsx     ', df_esc.shape)
print(' • servidores_limpo.xlsx  ', df_ser.shape)


## 4 · Visualização rápida com matplotlib

Gráficos de **barras**, **linhas** e **pizza**, prontos para exportar em PNG. Eles serão
reaproveitados pelo relatório (`m3_relatorio_reportlab.py`) e pelo dashboard.

### 4.1 Barras — Top 10 municípios por população

In [ ]:
top = df_mun.nlargest(10, 'População').sort_values('População')
fig, ax = plt.subplots()
ax.barh(top['Município'], top['População'], color='#2E7D32')
ax.set_xlabel('População')
ax.set_title('Top 10 municípios do Piauí por população')
fig.tight_layout()
fig.savefig(GRAFICOS / 'barras_populacao.png', dpi=120)
plt.show()
print('salvo: graficos/barras_populacao.png')


### 4.2 Pizza — distribuição por dependência administrativa

In [ ]:
dep = df_esc['Dependência'].value_counts()
fig, ax = plt.subplots()
ax.pie(dep.values, labels=dep.index, autopct='%1.0f%%', startangle=90)
ax.set_title('Escolas por dependência administrativa')
ax.axis('equal')
fig.tight_layout()
fig.savefig(GRAFICOS / 'pizza_dependencia.png', dpi=120)
plt.show()
print('salvo: graficos/pizza_dependencia.png')


### 4.3 Linhas — matrículas por município (ordenado)

Agrupamos matrículas por município e mostramos como linha — útil para ver queda/saltos.

In [ ]:
por_muni = (df_esc.groupby('Município')['Matrículas'].sum()
                .sort_values())
fig, ax = plt.subplots()
ax.plot(por_muni.index, por_muni.values, marker='o', color='#1565C0')
ax.set_ylabel('Matrículas')
ax.set_title('Total de matrículas por município')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
fig.savefig(GRAFICOS / 'linhas_matriculas.png', dpi=120)
plt.show()
print('salvo: graficos/linhas_matriculas.png')


### 4.4 >>> EXERCÍCIO DOS ALUNOS — Um gráfico a sua escolha <<<

Invente uma visualização nova que você considere útil.
Ideias: remuneração média por cargo, escolas por região, relação população×matrículas,
um scatter com Latitude/Longitude das escolas/servidores.

> **Dica de prompt para a IA**:
> "Crie um gráfico de dispersão (scatter) com a Latitude no eixo Y e Longitude no X,
> usando a base de municípios, colorindo por Região e salvando em graficos/scatter_municipios.png."

Implemente na célula abaixo e **confira** o PNG em `saida/modulo3/graficos/`.

In [ ]:
# TODO(aluno): crie um gráfico novo aqui e salve em GRAFICOS / 'meu_grafico.png'
# Exemplo de esqueleto:
# fig, ax = plt.subplots()
# ...
# fig.savefig(GRAFICOS / 'meu_grafico.png', dpi=120)
pass


---

## 5 · Fechamento

Você saiu com três planilhas **limpas** e três a quatro PNGs em `saida/modulo3/graficos/`.
Agora os scripts `m3_cruzamento.py`, `m3_agrupamentos.py`, `m3_diff_versoes.py`,
`m3_relatorio_reportlab.py` e `m3_dashboard_streamlit.py` têm tudo o que precisam.

### Lembrete — os 4 pilares voltam em cada script
  • **Decomposição** → cada script resolve um pedaço (cruzar, agrupar, comparar, relatar).
  • **Padrões**      → "ler → processar → conferir → salvar" se repete.
  • **Abstração**    → "base limpa" abstrai toda a sujeira que tínhamos aqui.
  • **Algoritmo**    → a ordem importa: ETL primeiro, análise depois.

### Tarefa aberta (contexto Piauí)
Escolha um **problema real** do seu município (ex.: "onde faltam escolas em relação à
população?", "qual região tem a menor remuneração média?") e produza, com essa base, um
**relatório consolidado**. Esse é o ponto de partida do projeto final do Módulo 3.
